# 03_grid_events_silver

## Purpose

Transform Bronze grid event records into a cleaned and reusable Silver table.

This notebook standardizes event fields, casts duration values, creates event-day fields, applies a severity-band UDF, filters invalid records, and writes the result to the Silver layer.

In [0]:
import sys
import importlib

repo_src_path = "/Workspace/Users/dirella@startsteps.org/vattenfall-week9-capstone-anna/src"

if repo_src_path not in sys.path:
    sys.path.append(repo_src_path)

import transforms.grid_event_transforms as grid_transforms
importlib.reload(grid_transforms)

from udfs.grid_udfs import classify_severity_band
from validation.silver_checks import print_basic_quality_summary

In [0]:
catalog = "vattenfall_dev"

bronze_table = f"{catalog}.raw.bronze_grid_events"
silver_table = f"{catalog}.refined.silver_grid_events"

print("Source Bronze table:", bronze_table)
print("Target Silver table:", silver_table)

In [0]:
bronze_grid_df = spark.table(bronze_table)

before_count = bronze_grid_df.count()

print("Rows before cleaning:", before_count)
display(bronze_grid_df.limit(20))

In [0]:
print(spark.table("vattenfall_dev.raw.bronze_grid_events").columns)

In [0]:
silver_grid_df = (
    bronze_grid_df
    .transform(grid_transforms.standardize_grid_event_columns)
    .transform(grid_transforms.cast_grid_event_fields)
    .transform(grid_transforms.add_grid_event_day)
    .withColumn("severity_band", classify_severity_band("severity"))
    .transform(grid_transforms.filter_invalid_grid_events)
)

after_count = silver_grid_df.count()

print("Rows after cleaning:", after_count)
print("Rows removed:", before_count - after_count)

display(silver_grid_df.limit(20))

In [0]:
(
    silver_grid_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_table)
)

print("Written Silver table:", silver_table)

In [0]:
silver_df = spark.table(silver_table)

print_basic_quality_summary(silver_df, silver_table)

display(silver_df.limit(20))

In [0]:
print("Severity values:")
silver_df.select("severity").distinct().show(truncate=False)

print("Severity band values:")
silver_df.select("severity_band").distinct().show(truncate=False)

print("Null check for key Silver fields:")
for column_name in ["event_id", "region", "asset_id", "event_type", "severity", "severity_band", "duration_minutes", "event_day"]:
    null_count = silver_df.filter(f"{column_name} IS NULL").count()
    print(column_name, "nulls:", null_count)

print("Invalid duration rows:")
invalid_duration_count = silver_df.filter("duration_minutes < 0").count()
print("duration_minutes < 0:", invalid_duration_count)